# Avito: кандидатогенерация для поиска услуг — EDA и бейзлайн

**Задача:** для каждого запроса из `benchmark_queries.parquet` вернуть до 50 `item_id` из корпуса; метрика — Recall@50.

**Что делает ноутбук**
1. Загружает данные и отвечает на ключевые вопросы EDA (от них зависят решения ниже).
2. Строит локальную валидацию — псевдо-бенчмарк из train с той же долей «знакомых» запросов, что и в бенчмарке.
3. Бейзлайн: пул кандидатов из нескольких генераторов (BM25 по полям, покрытие лемм, P(микрокатегория | запрос), локация, «память» train) + линейная формула с весами, подобранными на валидации.
4. Сохраняет `answer.csv`, проверяет формат и печатает md5 для контроля воспроизводимости.

**Запуск**
* Kaggle: подключить датасет с данными (путь ищется автоматически в `/kaggle/input`), включить Internet (клонирование репозитория и `pip install pymorphy3`). GPU не нужен.
* Локально: `pip install -r requirements.txt`, положить три parquet-файла в `./data` (или задать `DATA_DIR`), запустить ноутбук из папки `notebooks/`.

**Используемые open-source библиотеки:** numpy, pandas, scipy, scikit-learn (`CountVectorizer`), pyarrow, pymorphy3 (лемматизация). Внешних API нет.

## 0. Окружение и воспроизводимость

In [ ]:
import os, sys, subprocess
from pathlib import Path

# Число потоков BLAS фиксируется ДО импорта numpy — иначе настройка не действует
N_THREADS = 4
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = str(N_THREADS)

IS_KAGGLE = Path("/kaggle/input").exists()
REPO_URL = "https://github.com/mishin-mikhail/avito_autumn_dev.git"
# Ветка или хеш коммита. Для финального прогона — конкретный хеш, чтобы код не «уехал»
REPO_REF = "main"


def _github_token():
    """Токен для приватного репозитория: переменная GITHUB_TOKEN или Kaggle Secret с тем же именем."""
    tok = os.environ.get("GITHUB_TOKEN")
    if tok:
        return tok
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:  # не Kaggle или секрет не подключён — пробуем без токена
        return None


def _git(args, token=None) -> str:
    """git без утечки токена: он передаётся через переменные окружения (не в аргументах
    и не в .git/config), а в тексте ошибки маскируется."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        import base64
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1",
                   GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    r = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if r.returncode != 0:
        err = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError(f"git {args[0] if args[0] != '-C' else args[2]} завершился с ошибкой:\n{err}")
    return r.stdout.strip()


def find_repo_root() -> Path:
    """Корень репозитория: локально — рядом с ноутбуком; на Kaggle — датасет с кодом или git clone."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "candidates.py").exists():
            return p
    if IS_KAGGLE:
        hits = sorted(Path("/kaggle/input").rglob("src/candidates.py"))
        if hits:
            return hits[0].parents[1]
        dst = Path("/kaggle/working/avito-candgen")
        token = _github_token()
        if not dst.exists():
            _git(["clone", "--quiet", REPO_URL, str(dst)], token)
        _git(["-C", str(dst), "fetch", "--quiet", "origin"], token)
        # REPO_REF может быть веткой (берём её свежую версию) или хешем коммита
        branch = subprocess.run(["git", "-C", str(dst), "rev-parse", "--verify", "--quiet",
                                 f"origin/{REPO_REF}"], capture_output=True).returncode == 0
        _git(["-C", str(dst), "checkout", "--quiet", "--force", "--detach",
              f"origin/{REPO_REF}" if branch else REPO_REF])
        print("commit:", _git(["-C", str(dst), "rev-parse", "HEAD"]))
        if not (dst / "src" / "candidates.py").exists():
            raise RuntimeError(
                "В корне репозитория нет src/candidates.py — скорее всего, код лежит во вложенной папке. "
                f"Содержимое корня: {sorted(x.name for x in dst.iterdir())}")
        return dst
    raise RuntimeError("Не найден код решения (папка src). Запустите ноутбук внутри репозитория.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT))

# pymorphy3 нет в образе Kaggle — ставим закреплённые версии (локально — из requirements.txt)
try:
    import pymorphy3  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pymorphy3==2.0.6", "pymorphy3-dicts-ru==2.4.417150.4580142"], check=True)
print("repo:", REPO_ROOT, "| kaggle:", IS_KAGGLE)

In [ ]:
import json, time
from contextlib import contextmanager

import numpy as np
import pandas as pd

from src.config import CFG
from src.repro import seed_everything, library_versions, file_md5
from src.paths import get_data_dir, get_work_dir, get_output_dir
from src.text import Lemmatizer
from src.data import load_train, load_benchmark, load_train_items_text
from src.validation import (seen_share, build_validation_split, build_val_queries,
                            recall_at_k, recall_by_segment)
from src.priors import MicrocatPrior, ItemStats
from src.candidates import (Vocab, build_corpus, build_queries, generate_pool,
                            attach_labels, source_recall, FEATURES)
from src.tuning import coordinate_ascent, pool_recall, linear_score, predict_from_pool
from src.submit import save_answer, validate_answer

assert CFG.n_threads == N_THREADS
seed_everything(CFG.seed)

DATA_DIR, WORK_DIR, OUT_DIR = get_data_dir(), get_work_dir(), get_output_dir()
VERSIONS = library_versions()
pd.set_option("display.max_colwidth", 120)
print("data:", DATA_DIR, "\nwork:", WORK_DIR, "\nout: ", OUT_DIR)
print(VERSIONS)


@contextmanager
def timer(name):
    t = time.perf_counter()
    yield
    print(f"[{name}] {time.perf_counter() - t:.1f} c")

## 1. Загрузка данных

Описания объявлений из train не грузим целиком: они нужны только для позитивов валидации и дочитываются точечно (фильтр pyarrow по `item_id`). Все id приводятся к строкам.

In [ ]:
with timer("load"):
    train = load_train(DATA_DIR)
    bench_q, bench_items = load_benchmark(DATA_DIR)

print("train:", train.shape, "| bench queries:", bench_q.shape, "| bench items:", bench_items.shape)
print(f"память train ≈ {train.memory_usage(deep=True).sum() / 2**30:.2f} ГБ")
train.head(3)

In [ ]:
# Пропуски и типы — чтобы сразу увидеть сюрпризы в схеме
pd.DataFrame({
    "dtype_train": train.dtypes.astype(str),
    "na_train": train.isna().mean().round(4),
}).join(pd.DataFrame({"na_bench_items": bench_items.isna().mean().round(4)}), how="outer")

## 2. EDA: ответы на вопросы, от которых зависит решение

«Запрос» = группа строк train с одинаковым `query_key` (нормализованный текст + локация + доставка + фильтры + категория поиска).

In [ ]:
# 2.1 Сколько выбранных объявлений на запрос (в бенчмарке «обычно 1–2»)
g = train.groupby("query_key")["item_id"].nunique()
print(f"групп-запросов в train: {len(g):,}; уникальных текстов: {train['norm_text'].nunique():,}; "
      f"уникальных объявлений: {train['item_id'].nunique():,}")
print(g.describe(percentiles=[.5, .75, .9, .99]).round(2).to_string())
g.clip(upper=10).value_counts().sort_index().rename("групп")

In [ ]:
# 2.2 Роль локации: доля пар, где локация объявления совпадает с локацией поиска
same_loc = train["search_location_id"].to_numpy() == train["item_location_id"].to_numpy()
print(train.assign(same_loc=same_loc)
      .groupby("search_is_delivery_search")["same_loc"].agg(["size", "mean"]).round(4))

# есть ли у локаций бенчмарка объявления в корпусе
bench_locs_in_corpus = bench_q["search_location_id"].isin(pd.Index(bench_items["item_location_id"].unique()))
print(f"запросов бенчмарка, у чьей локации есть объявления в корпусе: {bench_locs_in_corpus.mean():.3f}")

In [ ]:
# 2.3 Пересекаются ли объявления корпуса с train (решает, полезны ли статистики по item_id)
train_item_index = pd.Index(train["item_id"].unique())
overlap_bench = float(bench_items["item_id"].isin(train_item_index).mean())
overlap_pairs = float(train["item_id"].isin(pd.Index(bench_items["item_id"])).mean())
USE_ITEM_STATS = overlap_bench >= CFG.item_stats_min_overlap
print(f"доля объявлений корпуса, встречающихся в train: {overlap_bench:.3f}")
print(f"доля пар train, чьё объявление есть в корпусе:   {overlap_pairs:.3f}")
print("используем статистики по item_id:", USE_ITEM_STATS)

In [ ]:
# 2.4 Насколько запросы бенчмарка «знакомы» train
BENCH_SEEN = seen_share(bench_q["norm_text"], train["norm_text"])
bench_key_seen = float(bench_q["query_key"].isin(pd.Index(train["query_key"].unique())).mean())
print(f"текст запроса бенчмарка встречается в train: {BENCH_SEEN:.3f}")
print(f"полный query_key встречается в train:        {bench_key_seen:.3f}")
bench_q[["search_query", "search_infm_params_text", "search_category"]].sample(10, random_state=CFG.seed)

In [ ]:
# 2.5 Фильтры: какие бывают и соблюдаются ли в train
print("доля запросов с фильтрами: train "
      f"{(train['filters_norm'] != '').mean():.3f} | bench {(bench_q['filters_norm'] != '').mean():.3f}")
print("доставка: train "
      f"{train['search_is_delivery_search'].mean():.3f} | bench {bench_q['search_is_delivery_search'].mean():.3f}")
display(train["search_infm_params_text"].value_counts().head(15))

has_thr = ~np.isnan(train["rating_thr"].to_numpy())
if has_thr.any():
    r = train.loc[has_thr, "item_rating"].to_numpy()
    t = train.loc[has_thr, "rating_thr"].to_numpy()
    print(f"пар с фильтром по рейтингу: {has_thr.mean():.3f}; из них рейтинг ≥ порога: "
          f"{np.mean(r >= t):.3f}, рейтинг пуст: {np.mean(np.isnan(r)):.3f}")

In [ ]:
# 2.6 Категории: насколько категория поиска определяет категорию объявления
print("категорий поиска:", train["search_category"].nunique(),
      "| категорий объявлений:", train["item_category_id"].nunique(),
      "| микрокатегорий:", train["item_microcat_id"].nunique())
ct = train.groupby(["search_category", "item_category_id"]).size()
purity = (ct.groupby(level=0).max() / ct.groupby(level=0).sum())
print(f"средняя доля самой частой item-категории внутри категории поиска: "
      f"{np.average(purity, weights=ct.groupby(level=0).sum()):.3f}")
train["search_category"].value_counts().head(10)

**Как EDA используется дальше** (автоматически, без ручных констант):
* `BENCH_SEEN` задаёт долю «знакомых» запросов в валидации;
* `USE_ITEM_STATS` включает/выключает признаки популярности и «памяти» по `item_id`;
* роль локации, фильтров и категорий не зашита жёстко, а учитывается признаками с весами, подобранными на валидации.

## 3. Лемматизация запросов

Лемматизируем уникальные тексты и раскладываем обратно — train содержит много повторов.

In [ ]:
lem = Lemmatizer()
with timer("lemmatize train queries"):
    uq = pd.unique(train["search_query"].to_numpy(dtype=object))
    key_of = dict(zip(uq, lem.key_many(uq)))
    train["lemma_key"] = train["search_query"].map(key_of)
print("уникальных текстов:", len(uq), "| слов в кеше лемм:", lem.cache_size)
train[["search_query", "lemma_key"]].drop_duplicates().head(8)

## 4. Локальная валидация

Разбиение детерминировано (md5 от ключа с солью `CFG.val_salt`). Доля «знакомых» текстов подгоняется под бенчмарк, а объявления валидационных запросов подмешиваются в **реальный корпус бенчмарка** — так распределение «чужих» объявлений совпадает с боевым.

In [ ]:
with timer("split"):
    val_keys, fold_mask, split_info = build_validation_split(
        train, BENCH_SEEN, CFG.n_val_queries, CFG.text_holdout_frac, CFG.val_salt)
    val_q, val_truth = build_val_queries(train, val_keys)
    train_fold = train[fold_mask].reset_index(drop=True)

val_q["seen"] = val_q["norm_text"].isin(pd.Index(train_fold["norm_text"].unique()))
print(json.dumps(split_info, ensure_ascii=False, indent=1))
print(f"доля знакомых: валидация {val_q['seen'].mean():.3f} vs бенчмарк {BENCH_SEEN:.3f}")
print("позитивов на запрос:", pd.Series([len(t) for t in val_truth]).describe().round(2).to_dict())

In [ ]:
# Корпус валидации = корпус бенчмарка + недостающие позитивы валидации (с описаниями из train)
val_pos_ids = sorted(dict.fromkeys(i for rel in val_truth for i in rel))
bench_id_set = frozenset(bench_items["item_id"])      # только для проверки вхождения
missing = [i for i in val_pos_ids if i not in bench_id_set]
with timer("load val positives"):
    extra = load_train_items_text(DATA_DIR, missing)
val_items = pd.concat([bench_items, extra[bench_items.columns.intersection(extra.columns)]],
                      ignore_index=True)
print(f"корпус валидации: {len(val_items):,} (добавлено позитивов: {len(extra):,} из {len(val_pos_ids):,})")

# Общие словари локаций и микрокатегорий (объединение всех источников → нет неизвестных значений)
loc_vocab = Vocab(pd.concat([train["search_location_id"], train["item_location_id"],
                             bench_q["search_location_id"], val_items["item_location_id"]]).tolist())
micro_vocab = Vocab(pd.concat([train["item_microcat_id"], val_items["item_microcat_id"]]).tolist())
print("локаций:", len(loc_vocab), "| микрокатегорий:", len(micro_vocab))

## 5. Бейзлайн на валидации

Статистики (P(микрокатегория | запрос), популярность) считаются **только по train-фолду** — иначе утечка.

In [ ]:
def fit_stats(rows: pd.DataFrame, corpus):
    prior = MicrocatPrior(CFG.prior_alpha, CFG.prior_beta).fit(
        rows["lemma_key"], rows["search_category"],
        micro_vocab.encode(rows["item_microcat_id"].tolist(), missing=0), len(micro_vocab))
    stats = None
    if USE_ITEM_STATS:
        idx_of = {v: i for i, v in enumerate(corpus.item_ids)}
        item_idx = np.fromiter((idx_of.get(v, -1) for v in rows["item_id"]), np.int64, len(rows))
        stats = ItemStats().fit(rows["lemma_key"], item_idx, corpus.n)
        corpus.log_pop = np.log1p(stats.pop).astype(np.float32)
    return prior, stats


with timer("val corpus index"):
    val_corpus = build_corpus(val_items, lem, CFG, loc_vocab, micro_vocab)
with timer("val stats"):
    val_prior, val_stats = fit_stats(train_fold, val_corpus)
with timer("val queries"):
    val_qs = build_queries(val_q, lem, loc_vocab, val_prior)
with timer("val pool"):
    val_pool = generate_pool(val_corpus, val_qs, CFG, val_stats)
    val_pool = attach_labels(val_pool, val_corpus, val_truth)

n_rel = np.array([len(t) for t in val_truth], dtype=np.float64)
print(f"строк в пуле: {len(val_pool):,}")
source_recall(val_pool, n_rel).round(4)

`recall` у списков — полнота при их собственном K (сотни кандидатов), `pool` — потолок, который может достичь любое переранжирование пула. Теперь выбираем 50 из пула.

In [ ]:
Fv = val_pool[FEATURES].to_numpy(np.float64)
qv = val_pool["q"].to_numpy(np.int64)
rv = val_corpus.rank[val_pool["item"].to_numpy()]
lv = val_pool["label"].to_numpy(np.float64)
K, DEC = CFG.top_k, CFG.score_decimals


def weights(**kw):
    return np.array([kw.get(f, 0.0) for f in FEATURES], dtype=np.float64)


BASELINES = {
    "text (BM25 полей + покрытие)": weights(title=1, params=0.5, desc=0.3, cov=1),
    "text + локация": weights(title=1, params=0.5, desc=0.3, cov=1, loc=1),
    "text + локация + prior": weights(title=1, params=0.5, desc=0.3, cov=1, loc=1, logp=0.1),
}
for name, w in BASELINES.items():
    print(f"{name:35s} recall@{K} = {pool_recall(qv, rv, lv, n_rel, linear_score(Fv, w), K, DEC):.4f}")

In [ ]:
# Подбор весов координатным спуском. Сначала честная оценка: учим на половине запросов, меряем на другой
W0 = BASELINES["text + локация + prior"]
q_half = (np.arange(len(val_q)) % 2 == 1)   # запросы идут в md5-порядке, так что половины случайны
cv = {}
for name, tr_q in (("A→B", ~q_half), ("B→A", q_half)):
    tr, te = tr_q[qv], ~tr_q[qv]            # маски строк пула
    w_half, _ = coordinate_ascent(Fv[tr], FEATURES, qv[tr], rv[tr], lv[tr], n_rel * tr_q,
                                  W0, CFG.tune_grid, CFG.tune_passes, K, DEC, verbose=False)
    cv[name] = pool_recall(qv[te], rv[te], lv[te], n_rel * ~tr_q, linear_score(Fv[te], w_half), K, DEC)
    print(f"{name}: recall@{K} на отложенной половине = {cv[name]:.4f}")

In [ ]:
# Финальные веса — на всей валидации
with timer("tuning"):
    W, val_best = coordinate_ascent(Fv, FEATURES, qv, rv, lv, n_rel, W0,
                                    CFG.tune_grid, CFG.tune_passes, K, DEC)
pd.Series(W, index=FEATURES).rename("weight").to_frame().T

In [ ]:
# Контроль: метрика, посчитанная «как на платформе» по спискам item_id, совпадает с быстрой оценкой
fallback_val = np.lexsort((val_corpus.rank, -val_corpus.log_reviews, -val_corpus.log_pop))
val_pred = predict_from_pool(val_pool, val_corpus, FEATURES, W, K, DEC, len(val_q), fallback_val)
val_recall = recall_at_k(val_pred, val_truth, K)
print(f"Recall@{K} (валидация, in-sample веса): {val_recall:.4f} | CV по половинам: "
      + ", ".join(f"{k}={v:.4f}" for k, v in cv.items()))
assert abs(val_recall - val_best) < 1e-9

seg = [("знакомый" if s else "новый") + " / " + ("доставка" if d == 1 else "локально")
       for s, d in zip(val_q["seen"], val_q["search_is_delivery_search"])]
recall_by_segment(val_pred, val_truth, seg, K).round(4)

In [ ]:
# Разбор ошибок: запросы, где эталон не попал в топ-50, и где он был в пуле, но проиграл
miss = [i for i, (p, t) in enumerate(zip(val_pred, val_truth)) if not frozenset(t) & frozenset(p)]
in_pool = val_pool[val_pool["label"] == 1].groupby("q").size()
print(f"запросов без единого попадания: {len(miss)} ({len(miss) / len(val_q):.3f}); "
      f"из них эталон был в пуле: {np.isin(miss, in_pool.index).mean():.3f}")
idx_of_val = {v: i for i, v in enumerate(val_corpus.item_ids)}
examples = []
for qi in miss[:15]:
    it = val_items.iloc[idx_of_val[val_truth[qi][0]]]
    examples.append({"query": val_q.at[qi, "search_query"], "filters": val_q.at[qi, "search_infm_params_text"],
                     "deliv": val_q.at[qi, "search_is_delivery_search"],
                     "same_loc": it["item_location_id"] == val_q.at[qi, "search_location_id"],
                     "gold_title": it["item_title_raw"], "gold_params": it["item_infm_params_text"]})
pd.DataFrame(examples)

## 6. Предсказание для бенчмарка

Статистики пересчитываются по **всему** train, веса берутся с валидации.

In [ ]:
with timer("bench corpus index"):
    bench_corpus = build_corpus(bench_items, lem, CFG, loc_vocab, micro_vocab)
with timer("bench stats"):
    bench_prior, bench_stats = fit_stats(train, bench_corpus)
with timer("bench queries"):
    bench_qs = build_queries(bench_q, lem, loc_vocab, bench_prior)
with timer("bench pool"):
    bench_pool = generate_pool(bench_corpus, bench_qs, CFG, bench_stats)

fallback_bench = np.lexsort((bench_corpus.rank, -bench_corpus.log_reviews, -bench_corpus.log_pop))
bench_pred = predict_from_pool(bench_pool, bench_corpus, FEATURES, W, K, DEC, len(bench_q), fallback_bench)

answer_path = OUT_DIR / "answer.csv"
save_answer(bench_q["query_id"], bench_pred, answer_path)
check = validate_answer(answer_path, bench_q["query_id"], bench_items["item_id"], K)
print(check)
pd.read_csv(answer_path, dtype=str).head(3)

## 7. Артефакты и контроль воспроизводимости

In [ ]:
report = {
    "answer_md5": check["md5"],
    "val_recall@50_in_sample": val_recall,
    "val_recall@50_half_cv": cv,
    "val_pool_recall": source_recall(val_pool, n_rel)["recall"].to_dict(),
    "weights": dict(zip(FEATURES, map(float, W))),
    "use_item_stats": bool(USE_ITEM_STATS),
    "bench_seen_share": BENCH_SEEN,
    "split": split_info,
    "config": CFG.as_dict(),
    "versions": VERSIONS,
}
(WORK_DIR / "baseline_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=1))
# Пулы с признаками пригодятся для обучения ранкера на следующем этапе
val_pool.to_parquet(WORK_DIR / "val_pool_baseline.parquet", index=False)
bench_pool.to_parquet(WORK_DIR / "bench_pool_baseline.parquet", index=False)
val_q.drop(columns=["query_key"]).assign(truth=[" ".join(t) for t in val_truth]).to_parquet(
    WORK_DIR / "val_queries.parquet", index=False)

print("answer.csv md5:", check["md5"])
print("Повторный запуск ноутбука должен дать тот же md5.")